In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2025-07-05 11:11:40--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8001::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.1s    

2025-07-05 11:11:41 (8.42 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

len(text)

1115394

In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
# extract all the characters
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
# character encoding and decoding
itos = {i:ch for i,ch in enumerate(chars)}
stoi = {ch:i for i,ch in enumerate(chars)}
encode = lambda s : [stoi[c] for c in s]
decode = lambda l : ''.join([itos[i] for i in l])
print(encode("Hey there !!"))
print(decode(encode("Hey everyone")))

[20, 43, 63, 1, 58, 46, 43, 56, 43, 1, 2, 2]
Hey everyone


In [7]:
# encode the whole data
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [8]:
# Split dataset into training and validation
n = int(0.9*len(data))
train_data = data[:n]
validation_data = data[n:]

In [ ]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else validation_data
    idxs = torch.randint(len(data)-block_size, (batch_size,))
    x = torch.stack([data[idx:idx+block_size] for idx in idxs])
    y = torch.stack([data[idx+1:idx+block_size+1] for idx in idxs])
    return x, y

xb, yb = get_batch('train')
print(xb.shape)
print(xb)
print(yb.shape)
print(yb)

torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [ ]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        
        self.token_embedding_layer = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_layer(idx)
        loss = None

        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            x = logits[:, -1, :]
            probs = F.softmax(x, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.concat((idx, idx_next), dim=1)
        
        return idx


model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

idx = torch.zeros((1, 1), dtype=torch.long)
output = model.generate(idx, 100)
print(decode(output[0].tolist()))

torch.Size([32, 65])
tensor(4.7412, grad_fn=<NllLossBackward0>)

;MUJyHCfdqA:Y.ZYTM;aPDYBM.upKR rWEfVoFdUUP
f
I,wOO;T
EyWBzntmi&:ZDDrbYx!nuDMuSxuzKU$r-iKHOxufSQLLfBc


In [43]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [47]:
batch_size = 32

for i in range(10000):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


2.3828608989715576


In [50]:
idx = torch.zeros((1, 1), dtype=torch.long)
output = model.generate(idx, 500)
print(decode(output[0].tolist()))



Coford ndfle d pove ody d-ses LO:
's wasmbls isulacy;
WAd,
KERKE:
mawhelle I thew cods l,
Sometie du tshewecry hessbeleme pe, to t dsin:
GRortin ce motoovee y.
Hen athaurvardin, ind ck.
BRLI sonds tce
Yof I cey? pend off,
NSTh.
G st caney drs keth'l, hth t it t, wicow pay.

YBYomarsthilldwir wabst be ars st lsoonk-ve.
LABu herexVE:
CIZy hing
To haldsthorndot
Ano our at fofe?
V:
IFLus n
INGorthath s LAp
ANGorot,
D k id hakis denth ldo te cl be
Busth icNToss igay n:
SAricinu u t thued be s;
Dr, y


In [52]:
torch.backends.mps.is_available()

True

In [70]:
B, T, C = 4, 8, 32
x = torch.rand((B, T, C))

head_size = 16
key = torch.nn.Linear(C, head_size)
query = torch.nn.Linear(C, head_size)
value = torch.nn.Linear(C, head_size)
k = key(x)
q = query(x)
v = value(x)

affinities = q @ k.transpose(-2, -1)

tril = torch.tril(torch.ones((T, T)))
# affinities = torch.zeros((T, T))
affinities = affinities.masked_fill(tril == 0, float('-inf'))
affinities = torch.softmax(affinities, 2)

out = affinities @ v
affinities

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5984, 0.4016, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5337, 0.2937, 0.1726, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4487, 0.2511, 0.1660, 0.1342, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2542, 0.2164, 0.1591, 0.1248, 0.2456, 0.0000, 0.0000, 0.0000],
         [0.2844, 0.1738, 0.0979, 0.0664, 0.1962, 0.1811, 0.0000, 0.0000],
         [0.2227, 0.1391, 0.0933, 0.0788, 0.1395, 0.1657, 0.1609, 0.0000],
         [0.1948, 0.1229, 0.0678, 0.0433, 0.1574, 0.1435, 0.1719, 0.0983]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.6041, 0.3959, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3524, 0.2923, 0.3553, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2524, 0.2430, 0.3496, 0.1550, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2750, 0.1927, 0.2347, 0.1347, 0.1629, 0.0000, 0.0000, 0.0000],
         [0.2068, 0.176

In [73]:
key.out_features

16

In [82]:
a = torch.randn((4, 6, 32))
b = torch.randn((32, 8, 4))
c = a @ b